# Материалы

`festim.Material` — ключевой компонент (класс) для определения теплофизических свойств материалов, в которых рассматривается перенос атомов изотопов водорода. Они включают в себя, например, коэффициент диффузии, теплопроводность, теплоёмкость, плотность и т.д. 


## Задание свойств материала

В простейшем случае, при известной истории изменения температуры материала, его свойства достаточно задать с помощью предэкспоненциального множителя коэффициента диффузии $D_0$ и энергии активации $E_D$. По умолчанию FESTIM предполагает, что коэффициент диффузии подчиняется закону Аррениуса:

$$
D = D_0 \exp\left(-\frac{E_D}{k_B T}\right),
$$

где $k_B$ — постоянная Больцмана в эВ/К, а $T$ — температура в К.


In [1]:
import festim as F

mat = F.Material(D_0=1.11e-6, E_D=0.4)  # м²/с, эВ

При учёте сохранения химического потенциала на границах материалов растворимость водорода можно задать с помощью предэкспоненциального множителя коэффициента растворимости `K_S_0`, энергии активации растворимости `E_K_S` и закона растворимости `solubility_law` (`"henry"` или `"sievert"`):


In [2]:
mat.K_S_0 = 1.0
mat.E_K_S = 3.0
mat.solubility_law = "sievert"

## Задание теплофизических свойств

Для материалов можно задавать теплофизические свойства, такие как теплопроводность, теплоёмкость и плотность. Простейший способ:


In [3]:
mat.thermal_conductivity = 10.0  # Вт/(м·К)
mat.heat_capacity = 500.0  # Дж/(кг·К)
mat.density = 8000.0  # кг/м³

Эти свойства также можно задавать как функции температуры:


In [4]:
import ufl

mat.thermal_conductivity = lambda T: 3 * T + 2 * ufl.exp(-20 * T)
mat.heat_capacity = lambda T: 4 * T + 8
mat.density = lambda T: 7 * T + 5

## Назначение материалов различным подобластям

В FESTIM материал назначается объёмной подобласти. В одномерной модели подобласть задаётся интервалом с помощью `F.VolumeSubdomain1D`.

Ниже область длиной $1\times10^{-3}$ м разделена на два слоя с разными материалами.

In [5]:
# Материалы
material_1 = F.Material(
    name="Материал 1",
    D_0=1.11e-6,  # м²/с
    E_D=0.4,      # эВ
)

material_2 = F.Material(
    name="Материал 2",
    D_0=2.0e-6,   # м²/с
    E_D=0.3,      # эВ
)

# Одномерные объёмные подобласти
left_layer = F.VolumeSubdomain1D(
    id=1,
    borders=[0.0, 5e-4],
    material=material_1,
)

right_layer = F.VolumeSubdomain1D(
    id=2,
    borders=[5e-4, 1e-3],
    material=material_2,
)

volume_subdomains = [left_layer, right_layer]


Таким же способом одномерную область можно разделить на любое число слоёв: для каждого интервала создаётся свой `VolumeSubdomain1D` и назначается соответствующий материал.